# Durable execution  持久执行
持久执行是一种技术，它允许流程或工作流在关键节点保存进度，从而实现暂停并在之后从中断处精确恢复。
> 如果将 LangGraph 与检查点机制结合使用，则已启用持久执行。可以随时暂停和恢复工作流，即使在中断或故障之后也是如此。为了充分利用持久执行，请确保工作流设计为确定性和幂等性的 ，并将任何副作用或不确定操作封装在任务中。
## Requirements  要求
1. 通过指定一个检查点来保存工作流进度，从而实现工作流的持久化 。
2. 执行工作流时，请指定线程标识符 。这将跟踪特定工作流实例的执行历史记录。
3. 将任何不确定操作（例如，随机数生成）或具有副作用的操作（例如，文件写入、API 调用）封装在任务中，以确保在工作流恢复时，这些操作不会针对本次运行重复执行，而是从持久层检索其结果。
## Determinism and consistent replay 确定性和一致性重放
- **避免重复工作** ：如果一个节点包含多个带有副作用的操作（例如，日志记录、文件写入或网络调用），请将每个操作封装在一个**单独的任务**中。这样可以确保在工作流恢复时，操作不会重复执行，并且其结果可以从持久层检索。
- **封装非确定性操作**： 将任何可能产生非确定性结果的代码（例如，随机数生成）封装在**任务或节点**中。这样可以确保在恢复运行时，工作流能够严格按照记录的步骤顺序执行，并获得相同的结果。
- **使用幂等操作** ：尽可能确保副作用（例如 API 调用、文件写入）是幂等的。这意味着，如果工作流中某个操作失败后重试，其效果将与首次执行时相同。这一点对于导致数据写入的操作尤为重要。如果某个任务启动但未能成功完成，工作流恢复后将重新运行该任务 ，并依赖已记录的结果来保持一致性。使用幂等键或验证现有结果，以避免意外重复，从而确保工作流执行流畅且可预测。

## Durability modes  耐久模式
- `"exit"` ：LangGraph 仅在图执行成功退出、出现错误退出或因人为中断而退出时才会保存更改。这为长时间运行的图提供了最佳性能，但也意味着中间状态不会被保存，因此您无法从执行过程中发生的系统故障（例如进程崩溃）中恢复。
- `"async"` ：LangGraph 在执行下一步操作的同时异步地持久化更改。这提供了良好的性能和持久性，但如果进程在执行过程中崩溃，则存在 LangGraph 无法写入检查点的风险。
- `"sync"` ：LangGraph 会在下一步开始之前同步持久化更改。这确保 LangGraph 在继续执行之前写入每个检查点，从而提供高持久性，但代价是一定的性能开销。

## Using tasks in nodes  在节点中使用任务
如果一个节点包含多个操作，你可能会发现将每个操作转换为一个任务比将操作重构为单独的节点更容易。

## Resuming workflows  恢复工作流程
在工作流中启用持久执行后，可以针对以下场景恢复执行：
- 暂停和恢复工作流： 使用中断函数可以在特定点暂停工作流，使用 Command 原语可以恢复工作流并更新其状态。
- 故障恢复： 发生异常（例如，LLM 提供程序中断）后，自动从上次成功的检查点恢复工作流。这需要使用相同的线程标识符执行工作流，并将输入值设置为 None

## Starting points for resuming workflows 恢复工作流程的起点
- 如果您使用的是状态图（Graph API） ，则起点是执行停止的节点的开头。
- 如果在节点内部进行子图调用，则起始点将是调用已停止子图的父节点。在子图内部，起始点将是执行停止的具体节点 。
- 如果您使用的是函数式 API，则起始点是执行停止的入口点的开头。